In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
from scipy.io import arff
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

# --- Torchvision ---
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.models import densenet121, DenseNet121_Weights

# --- Scikit-learn Utilities ---
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler

# --- EDA ---
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster 
from scipy.spatial.distance import squareform
from statsmodels.stats.outliers_influence import variance_inflation_factor

import umap

print("--- Imports Complete ---")

--- Imports Complete ---


In [3]:
cuda_available = torch.cuda.is_available()

print("=== GPU AVAILABILITY DIAGNOSTIC ===")
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    # Detailed GPU information for performance optimization
    print(f"\n=== GPU HARDWARE DETAILS ===")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU Device: {torch.cuda.current_device()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

    # Memory analysis - critical for batch size optimization
    print(f"\n=== MEMORY CONFIGURATION ===")
    memory_allocated = torch.cuda.memory_allocated(0)
    memory_reserved = torch.cuda.memory_reserved(0)
    total_memory = torch.cuda.get_device_properties(0).total_memory

    print(f"Total GPU Memory: {total_memory / 1024**3:.2f} GB")
    print(f"Currently Allocated: {memory_allocated / 1024**2:.2f} MB")
    print(f"Currently Reserved: {memory_reserved / 1024**2:.2f} MB")
    print(f"Available Memory: {(total_memory - memory_reserved) / 1024**3:.2f} GB")

    # CUDA version compatibility
    print(f"\n=== SOFTWARE VERSIONS ===")
    print(f"PyTorch Version: {torch.__version__}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"cuDNN Version: {torch.backends.cudnn.version()}")
    print(f"cuDNN Enabled: {torch.backends.cudnn.enabled}")

else:
    print("\n=== CPU-ONLY CONFIGURATION ===")
    print("GPU acceleration not available. Training will use CPU.")
    print("For large models, consider:")
    print("1. Cloud services (Google Colab, AWS, Azure)")
    print("2. CUDA-compatible GPU installation")
    print("3. Reduced model size and batch size for CPU training")

# Performance implications:
print(f"\n=== PERFORMANCE EXPECTATIONS ===")
if cuda_available:
    print("--- GPU acceleration available - expect 10-100x speedup for large models")
    print("--- Large batch sizes supported (limited by GPU memory)")
    print("--- Suitable for production-scale training")
else:
    print("--- CPU-only mode - expect slower training")
    print("--- Smaller batch sizes recommended")
    print("--- Consider GPU resources for larger experiments")

=== GPU AVAILABILITY DIAGNOSTIC ===
CUDA Available: True

=== GPU HARDWARE DETAILS ===
Number of GPUs: 1
Current GPU Device: 0
GPU Name: NVIDIA GeForce RTX 4050 Laptop GPU

=== MEMORY CONFIGURATION ===
Total GPU Memory: 6.00 GB
Currently Allocated: 0.00 MB
Currently Reserved: 0.00 MB
Available Memory: 6.00 GB

=== SOFTWARE VERSIONS ===
PyTorch Version: 2.10.0+cu130
CUDA Version: 13.0
cuDNN Version: 91200
cuDNN Enabled: True

=== PERFORMANCE EXPECTATIONS ===
--- GPU acceleration available - expect 10-100x speedup for large models
--- Large batch sizes supported (limited by GPU memory)
--- Suitable for production-scale training


In [4]:
train_raw=pd.read_csv("../data/raw/KDDTrain+.csv", header=None)
test_raw=pd.read_csv("../data/raw/KDDTest+.csv", header=None)

columns = [
'duration','protocol_type','service','flag','src_bytes','dst_bytes','land',
'wrong_fragment','urgent','hot','num_failed_logins','logged_in','num_compromised',
'root_shell','su_attempted','num_root','num_file_creations','num_shells',
'num_access_files','num_outbound_cmds','is_host_login','is_guest_login',
'count','srv_count','serror_rate','srv_serror_rate','rerror_rate',
'srv_rerror_rate','same_srv_rate','diff_srv_rate','srv_diff_host_rate',
'dst_host_count','dst_host_srv_count','dst_host_same_srv_rate',
'dst_host_diff_srv_rate','dst_host_same_src_port_rate',
'dst_host_srv_diff_host_rate','dst_host_serror_rate',
'dst_host_srv_serror_rate','dst_host_rerror_rate',
'dst_host_srv_rerror_rate','label','difficulty'
]

train_raw.columns=columns
test_raw.columns=columns

print(f" KDDTrain+  loaded: {train_raw.shape[0]:>7,} rows × {train_raw.shape[1]} cols")
print(f" KDDTest+   loaded: {test_raw.shape[0]:>7,} rows × {test_raw.shape[1]} cols")

 KDDTrain+  loaded: 125,973 rows × 43 cols
 KDDTest+   loaded:  22,543 rows × 43 cols


In [37]:
train_labels=train_raw[['label', 'difficulty']].copy()
test_labels=test_raw[['label', 'difficulty']].copy()

# Drop labels from working DataFrames
train_df=train_raw.drop(columns=['label', 'difficulty'])
test_df=test_raw.drop(columns=['label', 'difficulty'])

print("<--- Labels quarantined successfully --->")
print(f"   train_labels shape : {train_labels.shape}")
print(f"   test_labels shape  : {test_labels.shape}")

# Confirm feature-only DataFrames have exactly 41 columns
assert train_df.shape[1] == 41, "Feature count mismatch in train_df"
assert test_df.shape[1]  == 41, "Feature count mismatch in test_df"
print("\n<--- All feature DataFrames confirmed at 41 columns.--->")
print(f" Train DF  Created: {train_df.shape[0]:>7,} rows × {train_df.shape[1]} cols")
print(f" Test DF   Created: {test_df.shape[0]:>7,} rows × {test_df.shape[1]} cols")

<--- Labels quarantined successfully --->
   train_labels shape : (125973, 2)
   test_labels shape  : (22543, 2)

<--- All feature DataFrames confirmed at 41 columns.--->
 Train DF  Created: 125,973 rows × 41 cols
 Test DF   Created:  22,543 rows × 41 cols


In [6]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory confirmed: {PROCESSED_DIR.resolve()}")

Output directory confirmed: C:\Users\Hp\dev\python\genai\Unsupervised Learning Project\NSL-KDD Cybersec\data\processed


# Stage 4 - Preprocessing
## 4.1 - Duplicate Removal

In [9]:
train_before = len(train_df)
train_df_clean=train_df.drop_duplicates().reset_index(drop=True)
print(f"Train duplicates removed: {train_before - len(train_df_clean)}")
print(f"Train shape after dedup : {train_df_clean.shape}")

# Keeping Test DF same as we do not remove dupes from test
test_df_clean  = test_df.copy().reset_index(drop=True)

# Assigned labels to same indices
train_labels_aligned = train_labels.iloc[:len(train_df_clean)].reset_index(drop=True)

Train duplicates removed: 16
Train shape after dedup : (125957, 41)


## 4.2 - Redundancy Consolidation


In [11]:
# Instead of dropping redundant columns, we consolidate their signals to preserve semantic strength as inferred from the Hierarchial Tree

#Justifications:
# TBA

def apply_merge(df):
    dfc=df.copy()

    # Merge 1: serror cols using mean
    serror_cols = ['serror_rate', 'srv_serror_rate',
                   'dst_host_serror_rate', 'dst_host_srv_serror_rate']
    dfc["serror_rate_mean"]=dfc[serror_cols].mean(axis=1)
    dfc.drop(columns=serror_cols, inplace=True)

    # Merge 2: rerror cols using mean
    rerror_cols = ['rerror_rate', 'srv_rerror_rate',
                   'dst_host_rerror_rate', 'dst_host_srv_rerror_rate']
    dfc["rerror_rate_mean"]=dfc[rerror_cols].mean(axis=1)
    dfc.drop(columns=rerror_cols, inplace=True)

    # Merge 3: privilege escalation using max-pooling
    dfc["privilege_escalation_signal"]=dfc[["num_root","num_compromised"]].max(axis=1)
    dfc.drop(columns=["num_root","num_compromised"], inplace=True)

    # Merge 4: same_srv using difference
    dfc['same_srv_scope_delta']=(dfc['same_srv_rate'] - dfc['dst_host_same_srv_rate'])
    dfc.drop(columns=['dst_host_same_srv_rate'], inplace=True)

    return dfc

train_df_merged=apply_merge(train_df_clean)
test_df_merged=apply_merge(test_df_clean)

print(f"Shape after merges — train: {train_df_merged.shape}")
print(f"New features added: serror_rate_mean, rerror_rate_mean, privilege_escalation_signal, same_srv_scope_delta")
print(f"Features removed: 4 serror + 4 rerror + num_root/num_compromised + dst_host_same_srv_rate = 11 columns")

Shape after merges — train: (125957, 34)
New features added: serror_rate_mean, rerror_rate_mean, privilege_escalation_signal, same_srv_scope_delta
Features removed: 4 serror + 4 rerror + num_root/num_compromised + dst_host_same_srv_rate = 11 columns


## 4.3 - Categorical Encoding


In [18]:
# Three features, three different strategies, all justified from previous EDA

# Encoding 1 - protocol_type : OHE
# Reason: OHE is exact and lossless, producing 3 binary columns. 
# drop_first=False as we want to keep all three columns and this is not logistic
# dummy variables are hard-coded to remove problematic signals

def encode_protocol(df, fit_categories=None):
    dummies=pd.get_dummies(df['protocol_type'], prefix='proto', dtype=float)
    if fit_categories is not None:
        for col in fit_categories:
            if col not in dummies.columns:
                dummies[col]=0.0
        dummies=dummies[fit_categories]
    df=pd.concat([df.drop(columns=['protocol_type']), dummies], axis=1)
    return df, list(dummies.columns)

train_df_enc, proto_cats = encode_protocol(train_df_merged)
test_df_enc, _  = encode_protocol(test_df_merged, proto_cats)
print("protocol_type OHE columns:", proto_cats)
    
# Encoding 2 - Flags: OHE with rare-grouping
# Has 11 unique values with 3 (SF, S0, REJ) covering ~95% of data, 
# rare_flags are collapsed to 'flag_rare' before OHE
# Prevents near zero variance and preserves rare flag semantics,
# preventing noise for distance based detectors
COMMON_FLAGS={'SF', 'S0', 'REJ', 'RSTR', 'RSTO'}

def encode_flag(df, fit_categories=None):
    df=df.copy()
    df['flag_grouped']=df['flag'].apply(
        lambda x: x if x in COMMON_FLAGS else 'flag_rare'
    )
    dummies=pd.get_dummies(df['flag_grouped'], prefix='flag', dtype=float)
    if fit_categories is not None:
        for col in fit_categories:
            if col not in dummies.columns:
                dummies[col]=0.0
        dummies=dummies[fit_categories]
    df=df.drop(columns=['flag', 'flag_grouped'])
    df=pd.concat([df, dummies], axis=1)
    return df, list(dummies.columns)

train_df_enc, flag_cats = encode_flag(train_df_enc)
test_df_enc, _  = encode_flag(test_df_enc, flag_cats)
print(f"flag OHE columns: {flag_cats}")

# Encoding 3 - Services: Frequency Encoding + Rare indicator
# Unseens are mapped to 0.0
# Has over 70 values with long tail, frequency encoding maps each service to each other in the Training set only
# Fit on Train only, Test Set is transformed using train-set metrics.
# Additionally, we make a is_rare_service_flag, a binary indicator for 
# servicexflag combination which are near impossible in normal traffic 
# (as identified in Heatmap analysis - 3B). This engineered feature encodes anamoly architecture directly
RARE_COMBINATIONS = {
    ('http', 'SH'), ('http', 'RSTOS0'), ('http', 'OTH'), ('http', 'S3'),
    ('smtp', 'SH'), ('smtp', 'OTH'), ('smtp', 'RSTOS0'), ('smtp', 'REJ'),
    ('smtp', 'S3'), ('smtp', 'RSTR'), ('ftp_data', 'RSTOS0'),
    ('ftp_data', 'SH'), ('ftp_data', 'OTH'), ('ftp_data', 'S2'),
    ('ftp_data', 'RSTR'), ('other', 'S3'), ('other', 'OTH'), ('other', 'S2'),
    ('telnet', 'SH'), ('telnet', 'RSTOS0'), ('telnet', 'OTH'),
    ('finger', 'SH'), ('finger', 'RSTOS0'), ('finger', 'S3'),
    ('ftp', 'S2'), ('ftp', 'SH'), ('ftp', 'RSTOS0'), ('ftp', 'OTH'),
    ('auth', 'SH'), ('private', 'OTH')
}
service_freq=train_df_enc['service'].value_counts(normalize=True).to_dict()

# orig_df is the pre-encoded df with the 'flag' variable still there
def encode_service(df, freq_map, orig_df=None):
    df=df.copy()
    df['service_freq']=df['service'].map(freq_map).fillna(0.0)
    if orig_df is not None:
        df['is_rare_service_flag']=[
            1.0 if (s,f) in RARE_COMBINATIONS else 0.0
            for s,f in zip(orig_df['service'], orig_df['flag'])
        ]
    else:
        df['is_rare_service_flag']=0.0
    df.drop(columns=['service'], inplace=True)
    return df

train_df_enc=encode_service(train_df_enc, service_freq, train_df_merged)
test_df_enc=encode_service(test_df_enc, service_freq, test_df_merged)

print(f"service encoded: service_freq + is_rare_service_flag")
print(f"Shape after all encoding — train: {train_df_enc.shape}")
    
    

protocol_type OHE columns: ['proto_icmp', 'proto_tcp', 'proto_udp']
flag OHE columns: ['flag_REJ', 'flag_RSTO', 'flag_RSTR', 'flag_S0', 'flag_SF', 'flag_flag_rare']
service encoded: service_freq + is_rare_service_flag
Shape after all encoding — train: (125957, 42)


## 4.4 - Feature Group Definitions

In [21]:
# Defining feature groups for scaling pipeline
# these 5 groups are selected for their specific treatment based on 3A+3D Analysis
# Each group decision is made on their distributional properties

# Group A - Heavy-tailed continuous: log1p then RobustScaler
GROUP_A = ['src_bytes', 'dst_bytes', 'duration', 'srv_count', 'count']

# Group B - Zero-inflated sparse counts: zero-inflated log1p then RobustScaler
GROUP_B = ['privilege_escalation_signal', 'hot', 'num_file_creations', 'num_access_files']

# Group C - Ceiling-effect counts: RobustScaler only (log worsens)
GROUP_C = ['dst_host_count', 'dst_host_srv_count']

# Group D - Rate features bounded [0,1] + new merged rates + scope delta:
# RobustScaler only - log is inappropriate for bounded rates
GROUP_D = ['serror_rate_mean', 'rerror_rate_mean', 'same_srv_rate',
           'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_diff_srv_rate',
           'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
           'same_srv_scope_delta', 'service_freq']

# Group E - Binary/sparse behavioral: pass-through, no scaling
GROUP_E = ['root_shell', 'land', 'is_guest_login', 'is_host_login',
           'su_attempted', 'wrong_fragment', 'urgent', 'num_shells',
           'logged_in', 'num_outbound_cmds', 'is_rare_service_flag', "num_failed_logins"]

# Integrity Verification
all_assigned = set(GROUP_A + GROUP_B + GROUP_C + GROUP_D + GROUP_E)
all_numeric = set(train_df_enc.select_dtypes(include=[np.number]).columns)

# OHE columns are not in groups, they are already binary floats, pass-through
ohe_cols = set(proto_cats + flag_cats)
unassigned = all_numeric - all_assigned - ohe_cols

print(f"OHE columns (pass-through): {sorted(ohe_cols)}")
print(f"Unassigned numeric features: {sorted(unassigned)}")


OHE columns (pass-through): ['flag_REJ', 'flag_RSTO', 'flag_RSTR', 'flag_S0', 'flag_SF', 'flag_flag_rare', 'proto_icmp', 'proto_tcp', 'proto_udp']
Unassigned numeric features: []


## 4.5 - Scaling Pipeline

In [30]:
from sklearn.preprocessing import RobustScaler
robust=RobustScaler()
def zero_inflated_log1p(series):
    #Apply log1p only to non-zero values, preserving the zero mass
    return series.apply(lambda x: np.log1p(x) if x > 0 else 0.0)

def scale_groups(df, scalers=None, fit=False,
                 group_a=GROUP_A, group_b=GROUP_B,
                 group_c=GROUP_C, group_d=GROUP_D):
    df = df.copy(deep=True)
    if fit:
        scalers = {}

    # Group A: log1p → RobustScaler
    for col in group_a:
        df[col] = np.log1p(df[col])
    if fit:
        scalers['A'] = RobustScaler()
        df[group_a] = scalers['A'].fit_transform(df[group_a])
    else:
        df[group_a] = scalers['A'].transform(df[group_a])

    # Group B: zero-inflated log1p → RobustScaler
    for col in group_b:
        df[col] = zero_inflated_log1p(df[col])
    if fit:
        scalers['B'] = RobustScaler()
        df[group_b] = scalers['B'].fit_transform(df[group_b])
    else:
        df[group_b] = scalers['B'].transform(df[group_b])

    # Group C: RobustScaler only
    if fit:
        scalers['C'] = RobustScaler()
        df[group_c] = scalers['C'].fit_transform(df[group_c])
    else:
        df[group_c] = scalers['C'].transform(df[group_c])

    # Group D: RobustScaler only
    if fit:
        scalers['D'] = RobustScaler()
        df[group_d] = scalers['D'].fit_transform(df[group_d])
    else:
        df[group_d] = scalers['D'].transform(df[group_d])

    if fit:
        return df, scalers
    return df

# Fit on train, apply to all
df_train_scaled, fitted_scalers=scale_groups(train_df_enc, fit=True)

df_test_scaled =scale_groups(test_df_enc, scalers=fitted_scalers, fit=False)

print(f"Scaling complete.")
print(f"Final train shape: {df_train_scaled.shape}")
print(f"Final test shape:  {df_test_scaled.shape}")


Scaling complete.
Final train shape: (125957, 42)
Final test shape:  (22543, 42)


In [31]:
#Sanity
for name, df in [('train', df_train_scaled),
                 ('test', df_test_scaled)]:
    nans = df.isnull().sum().sum()
    print(f"  {name} NaN count: {nans}")

  train NaN count: 0
  test NaN count: 0


## 4.6 - Saving to CSV and Recoverability Test

In [56]:
import joblib

# Parquet preserves dtypes exactly as it is and is fast in reading bacl
# Needed at inference time.
df_train_scaled.to_csv(PROCESSED_DIR / "train_processed.csv", index=False)
df_test_scaled.to_csv(PROCESSED_DIR / "test_processed.csv", index=False)
train_labels_aligned.to_csv(PROCESSED_DIR / "train_labels.csv", index=False)
test_labels.to_csv(PROCESSED_DIR / "test_labels.csv", index=False)
# Save pipeline artifacts for recoverability
pipeline_artifacts={
    'scalers':fitted_scalers,
    'proto_cats':proto_cats,
    'flag_cats':flag_cats,
    'service_freq':service_freq,
    'group_A':GROUP_A,
    'group_B':GROUP_B,
    'group_C':GROUP_C,
    'group_D':GROUP_D,
    'group_E':GROUP_E,
    'rare_combos':RARE_COMBINATIONS,
    'common_flags':COMMON_FLAGS,
}

joblib.dump(pipeline_artifacts, PROCESSED_DIR / "pipeline_artifacts.joblib")

print("Saved:")
for f in sorted(PROCESSED_DIR.iterdir()):
    size_kb=f.stat().st_size / 1024
    print(f"  {f.name:<45} {size_kb:>8.1f} KB")


Saved:
  pipeline_artifacts.joblib                          4.7 KB
  test_labels.parquet                               87.2 KB
  test_processed.csv                              6084.7 KB
  test_processed.parquet                           975.8 KB
  train_labels.csv                                1399.8 KB
  train_labels.parquet                             396.0 KB
  train_processed.csv                            33956.4 KB
  train_processed.parquet                         5441.0 KB


In [2]:
# --- Verify recoverability 
# This cell simulates a fresh kernel restart, loading everything from disk
# and confirm shapes, dtypes, and value ranges are intact.

df_train_recovered=pd.read_csv(PROCESSED_DIR / "train_processed.csv")
df_test_recovered=pd.read_csv(PROCESSED_DIR / "test_processed.csv")
train_labels_rec=pd.read_csv(PROCESSED_DIR / "train_labels.csv")
artifacts_rec=joblib.load(PROCESSED_DIR / "pipeline_artifacts.joblib")

df_test_par=pd.read_parquet(PROCESSED_DIR / "train_processed.parquet")

print("Recovery check:")
print(f"  train  : {df_train_recovered.shape}  | NaNs: {df_train_recovered.isnull().sum().sum()}")
print(f"  test   : {df_test_recovered.shape}   | NaNs: {df_test_recovered.isnull().sum().sum()}")
print(f"  labels : {train_labels_rec.shape}")
print(f"  artifacts keys: {list(artifacts_rec.keys())}")
print("\nDtype sample (first 5 cols):")
print(df_train_recovered.dtypes.head())
print("\nValue range sample:")
print(df_train_recovered[GROUP_A].describe().loc[['min','max']].round(3))

NameError: name 'PROCESSED_DIR' is not defined